In [17]:
# -*- coding: utf-8 -*-
import os, re, json
import numpy as np
import pandas as pd

from pathlib import Path
from collections import defaultdict

from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_class_weight

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier

# ----------------------------------------------------
# CONFIG
# ----------------------------------------------------
CSV_PATH = "adni_merged_features.csv"  # <-- change if needed
OUT_DIR  = "/mnt/data/model_compare_outputs"
RANDOM_STATE = 42
N_SPLITS = 5
MAX_CAT_UNIQUES = 30  # columns with <=30 unique values (non-numeric) treated as categoricals

os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------------------------------
# 1) Load data
# ----------------------------------------------------
df = pd.read_csv(CSV_PATH, low_memory=False)

if "label" not in df.columns:
    raise SystemExit("No 'label' column found in the dataset.")

# Robust subject groups: attempt to extract RID from 'visit_id' prefix before first underscore
if "visit_id" not in df.columns:
    raise SystemExit("No 'visit_id' column found. You need visit-wise IDs to group by subject.")

groups = df["visit_id"].astype(str).str.split("_").str[0]

# Drop non-feature identifier columns if present
drop_cols = [c for c in ["visit_id","RID","PTID","ID","SITEID","USERDATE","update_stamp"] if c in df.columns]
X = df.drop(columns=["label"] + drop_cols, errors="ignore")
y = df["label"].astype(str)

# ----------------------------------------------------
# 2) Identify column types + defensive leak filtering (double-check)
# ----------------------------------------------------
# Exclude any diagnosis-like or QC/meta columns that might have slipped in
EXCLUDE_PATTERNS = re.compile(r"(DIAGNOSIS|^DX|DXNORM|DXMCI|DXAD|DXAPP|CONFID|MOTHET|OTHDEM|DSEV|HAS_QC_ERROR|SOURCE|DONE)$",
                              re.I)

leak_cols = [c for c in X.columns if EXCLUDE_PATTERNS.search(c)]
if leak_cols:
    X = X.drop(columns=leak_cols, errors="ignore")

# Decide categorical vs numeric
cat_cols, num_cols = [], []
for c in X.columns:
    if pd.api.types.is_numeric_dtype(X[c]):
        num_cols.append(c)
    else:
        # treat as categorical if low-cardinality non-numeric
        if X[c].nunique(dropna=True) <= MAX_CAT_UNIQUES:
            cat_cols.append(c)
        else:
            # Try coercion to numeric; if fails, drop as high-card text
            coerced = pd.to_numeric(X[c], errors="coerce")
            if coerced.notna().sum() >= 0.8 * len(coerced):
                X[c] = coerced
                num_cols.append(c)
            else:
                # too messy -> drop
                X = X.drop(columns=[c])
# Recompute after possible drops
all_cols = X.columns.tolist()
cat_cols = [c for c in cat_cols if c in all_cols]
num_cols = [c for c in num_cols if c in all_cols and c not in cat_cols]

# ----------------------------------------------------
# 3) Preprocessor (no leakage: fitted inside CV folds)
# ----------------------------------------------------
pre = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline(steps=[
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=True
)

# ----------------------------------------------------
# 4) Class weights (to balance CN/MCI/AD)
# ----------------------------------------------------
classes = np.sort(y.unique())
class_wts = compute_class_weight(class_weight="balanced", classes=classes, y=y)
class_weight_dict = {cls: wt for cls, wt in zip(classes, class_wts)}
print("Class weights:", class_weight_dict)

# Helper to pass weights to models that support 'class_weight'
def get_model_set():
    models = {
        "LogisticRegression": LogisticRegression(
            multi_class="multinomial", solver="lbfgs", max_iter=2000, class_weight=class_weight_dict,
            n_jobs=None  # lbfgs ignores n_jobs
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=400, max_depth=None, min_samples_leaf=2, random_state=RANDOM_STATE,
            class_weight=class_weight_dict, n_jobs=-1
        ),
        "GradientBoosting": GradientBoostingClassifier(
            random_state=RANDOM_STATE
            # no class_weight in native GBC; class imbalance handled by loss heuristics; still a good baseline
        ),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            learning_rate=0.06, max_iter=400, max_depth=8, l2_regularization=0.5, random_state=RANDOM_STATE
            # sklearn's HGB doesn't accept class_weight; imbalance handled reasonably; you can enable focal-loss style via custom libs
        ),
    }
    return models

# NOTE: If you have XGBoost/LightGBM installed, add here:
# from xgboost import XGBClassifier
# models["XGBoost"] = XGBClassifier(
#     n_estimators=700, max_depth=6, learning_rate=0.05, subsample=0.9, colsample_bytree=0.8,
#     eval_metric="mlogloss", reg_lambda=1.0, tree_method="hist", random_state=RANDOM_STATE,
#     num_class=len(classes)
# )
# For XGB, use 'fit_params={"sample_weight": ...}' per-class if needed.

# ----------------------------------------------------
# 5) Cross-validation with GroupKFold (by subject)
# ----------------------------------------------------
gkf = GroupKFold(n_splits=N_SPLITS)

def evaluate_model(name, estimator, X, y, groups):
    """
    Returns fold metrics and aggregate predictions for a model.
    """
    fold_rows = []
    oof_pred = pd.Series(index=y.index, dtype=object)
    oof_proba = np.zeros((len(y), len(classes)))  # if model supports predict_proba

    for fold, (tr, vl) in enumerate(gkf.split(X, y, groups), 1):
        Xtr, Xvl = X.iloc[tr], X.iloc[vl]
        ytr, yvl = y.iloc[tr], y.iloc[vl]

        pipe = Pipeline([("pre", pre), ("clf", estimator)])
        pipe.fit(Xtr, ytr)

        # Predictions
        yhat = pipe.predict(Xvl)
        oof_pred.iloc[vl] = yhat

        # Probabilities (if available)
        auc_macro = np.nan
        if hasattr(pipe.named_steps["clf"], "predict_proba"):
            P = pipe.predict_proba(Xvl)
            oof_proba[vl, :] = P
            # one-vs-rest AUC
            try:
                y_bin = pd.get_dummies(yvl).reindex(columns=classes, fill_value=0).values
                auc_macro = roc_auc_score(y_bin, P, average="macro")
            except Exception:
                pass

        acc = accuracy_score(yvl, yhat)
        f1m = f1_score(yvl, yhat, average="macro")
        # Per-class recall
        rep = classification_report(yvl, yhat, output_dict=True, zero_division=0)
        rec_cn  = rep.get("CN", {}).get("recall", np.nan)
        rec_mci = rep.get("MCI", {}).get("recall", np.nan)
        rec_ad  = rep.get("AD", {}).get("recall", np.nan)

        fold_rows.append({
            "model": name, "fold": fold, "n_val": len(vl),
            "macro_auc": auc_macro, "macro_f1": f1m, "accuracy": acc,
            "recall_CN": rec_cn, "recall_MCI": rec_mci, "recall_AD": rec_ad
        })

    # Aggregate confusion matrix
    cm = confusion_matrix(y, oof_pred, labels=classes)
    cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in classes], columns=[f"pred_{c}" for c in classes])
    return pd.DataFrame(fold_rows), cm_df

all_fold_metrics = []
cms = {}
models = get_model_set()

for name, est in models.items():
    print(f"\n===== Training {name} =====")
    folds_df, cm_df = evaluate_model(name, est, X, y, groups)
    all_fold_metrics.append(folds_df)
    cms[name] = cm_df

# ----------------------------------------------------
# 6) Summaries & exports
# ----------------------------------------------------
folds_all = pd.concat(all_fold_metrics, ignore_index=True)

summary = (
    folds_all.groupby("model")[["macro_auc","macro_f1","accuracy","recall_CN","recall_MCI","recall_AD"]]
    .agg(["mean","std"])
)
# Flatten columns
summary.columns = [f"{a}_{b}" for a,b in summary.columns]
summary = summary.reset_index().sort_values("macro_f1_mean", ascending=False)

# Save all
folds_all.to_csv(os.path.join(OUT_DIR, "cv_folds_diagnostics.csv"), index=False)
summary.to_csv(os.path.join(OUT_DIR, "model_comparison.csv"), index=False)

# Save confusion matrices
for name, cm_df in cms.items():
    cm_df.to_csv(os.path.join(OUT_DIR, f"confusion_matrix_overall_{name}.csv"))

print("\n=== Model comparison table ===")
print(summary)

print(f"\nSaved:\n- {os.path.join(OUT_DIR,'model_comparison.csv')}\n- {os.path.join(OUT_DIR,'cv_folds_diagnostics.csv')}\n- confusion matrices per model in {OUT_DIR}")


Class weights: {'AD': 1.7193808882907133, 'CN': 0.8300844704353476, 'MCI': 0.7870013861081164, 'nan': 1.060385972193401}

===== Training LogisticRegression =====


C:\Users\acer\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\acer\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stab


===== Training RandomForest =====

===== Training GradientBoosting =====

===== Training HistGradientBoosting =====


C:\Users\acer\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\acer\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\acer\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 501, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\acer\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 969, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\acer\AppData\Local\Programs\Python\Python310\lib\subproces


=== Model comparison table ===
                  model  macro_auc_mean  macro_auc_std  macro_f1_mean  \
1  HistGradientBoosting        0.993251       0.000884       0.939866   
0      GradientBoosting        0.990864       0.001286       0.932370   
3          RandomForest        0.988594       0.001851       0.921221   
2    LogisticRegression        0.962741       0.004705       0.847832   

   macro_f1_std  accuracy_mean  accuracy_std  recall_CN_mean  recall_CN_std  \
1      0.003915       0.938796      0.003588        0.940861       0.004546   
0      0.006221       0.930626      0.006006        0.937037       0.008154   
3      0.008942       0.919031      0.009211        0.929885       0.005484   
2      0.009134       0.840411      0.010733        0.856927       0.005176   

   recall_MCI_mean  recall_MCI_std  recall_AD_mean  recall_AD_std  
1         0.938167        0.008761        0.946371       0.013273  
0         0.926096        0.011285        0.937978       0.013829  
3 

In [18]:
import pandas as pd

df1 = pd.read_csv("adni_features_noleak.csv", low_memory=False)
# print(df1.columns.tolist())    # show all column names
# print(df.head()[0:])         # peek at first 3 rows


In [19]:
# ---- SHAP on GradientBoosting (global explanations) ----
try:
    import shap, matplotlib.pyplot as plt

    # Refit a full pipeline with GradientBoosting on ALL data
    gb_final = GradientBoostingClassifier(random_state=RANDOM_STATE)
    pipe_gb = Pipeline([("pre", pre), ("clf", gb_final)])
    pipe_gb.fit(X, y)

    # Transform X to get feature matrix and names
    X_trans = pipe_gb.named_steps["pre"].transform(X)
    feat_names = pipe_gb.named_steps["pre"].get_feature_names_out()

    # Subsample to keep SHAP fast
    rng = np.random.RandomState(42)
    n = min(1500, X_trans.shape[0])
    idx = rng.choice(X_trans.shape[0], size=n, replace=False)
    X_shap = X_trans[idx]

    # TreeExplainer works well with scikit GradientBoosting
    explainer = shap.TreeExplainer(pipe_gb.named_steps["clf"])
    shap_values = explainer.shap_values(X_shap)

    # Summary dot plot
    plt.figure()
    shap.summary_plot(shap_values, X_shap, feature_names=feat_names, show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "shap_summary_dot.png"), dpi=200)
    plt.close()

    # Summary bar plot
    plt.figure()
    shap.summary_plot(shap_values, X_shap, feature_names=feat_names, plot_type="bar", show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "shap_summary_bar.png"), dpi=200)
    plt.close()

    print(f"Saved SHAP plots: {os.path.join(OUT_DIR,'shap_summary_dot.png')}, "
          f"{os.path.join(OUT_DIR,'shap_summary_bar.png')}")
except Exception as e:
    print("SHAP generation skipped or failed:", e)


SHAP generation skipped or failed: GradientBoostingClassifier is only supported for binary classification right now!


In [2]:
# -*- coding: utf-8 -*-
import os, re
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.utils.class_weight import compute_class_weight

# -----------------------------
# CONFIG
# -----------------------------
CSV_PATH = "adni_merged_features.csv"   # <-- change if needed
OUT_DIR  = "/mnt/data/model_compare_outputs"
RANDOM_STATE = 42
MAX_CAT_UNIQUES = 30

os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# 1) Load + clean labels
# -----------------------------
df = pd.read_csv(CSV_PATH, low_memory=False)

if "label" not in df.columns:
    raise SystemExit("No 'label' column found in the dataset.")

if "visit_id" not in df.columns:
    raise SystemExit("No 'visit_id' column found in the dataset.")

# normalize label, drop unlabeled rows
df["label"] = df["label"].astype(str).str.strip()
df = df[~df["label"].isin(["", "nan", "NaN", "None"])]
df = df.dropna(subset=["label"])

# -----------------------------
# 2) Build X / y with leak filtering
# -----------------------------
drop_cols = [c for c in ["visit_id","RID","PTID","ID","SITEID","USERDATE","update_stamp"] if c in df.columns]

X = df.drop(columns=["label"] + drop_cols, errors="ignore")
y = df["label"].astype(str)

# Remove diagnosis-like and QC/meta columns
EXCLUDE_PATTERNS = re.compile(
    r"(DIAGNOSIS|^DX|DXNORM|DXMCI|DXAD|DXAPP|CONFID|MOTHET|OTHDEM|DSEV|HAS_QC_ERROR|SOURCE|DONE)$",
    re.I
)
leak_cols = [c for c in X.columns if EXCLUDE_PATTERNS.search(c)]
if leak_cols:
    X = X.drop(columns=leak_cols, errors="ignore")

# Decide categorical vs numeric (coerce where reasonable)
cat_cols, num_cols = [], []
for c in X.columns:
    if pd.api.types.is_numeric_dtype(X[c]):
        num_cols.append(c)
    else:
        # treat as categorical if low-cardinality
        if X[c].nunique(dropna=True) <= MAX_CAT_UNIQUES:
            cat_cols.append(c)
        else:
            # try coercion to numeric
            coerced = pd.to_numeric(X[c], errors="coerce")
            if coerced.notna().sum() >= 0.8 * len(coerced):
                X[c] = coerced
                num_cols.append(c)
            else:
                X = X.drop(columns=[c])

# re-check after drops
all_cols = X.columns.tolist()
cat_cols = [c for c in cat_cols if c in all_cols]
num_cols = [c for c in num_cols if c in all_cols and c not in cat_cols]

# Preprocessor
pre = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=True
)

# Class weights (CN/MCI/AD only)
classes = np.sort(y.unique())
class_wts = compute_class_weight(class_weight="balanced", classes=classes, y=y)
class_weight_dict = {cls: wt for cls, wt in zip(classes, class_wts)}
print("Class weights:", class_weight_dict)

# -----------------------------
# 3) SHAP for GradientBoosting (tree-based)
# -----------------------------
print("\nFitting GradientBoosting for SHAP...")
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
pipe_gb = Pipeline([("pre", pre), ("clf", gb)])
pipe_gb.fit(X, y)

# Transform features and names
X_trans_gb = pipe_gb.named_steps["pre"].transform(X)
feat_names_gb = pipe_gb.named_steps["pre"].get_feature_names_out()

# Subsample to keep SHAP fast
rng = np.random.RandomState(42)
n = min(1500, X_trans_gb.shape[0])
idx = rng.choice(X_trans_gb.shape[0], size=n, replace=False)
X_shap_gb = X_trans_gb[idx]

# SHAP with TreeExplainer
try:
    import shap, matplotlib.pyplot as plt
    explainer_gb = shap.TreeExplainer(pipe_gb.named_steps["clf"])
    shap_values_gb = explainer_gb.shap_values(X_shap_gb)

    # dot summary
    plt.figure()
    shap.summary_plot(shap_values_gb, X_shap_gb, feature_names=feat_names_gb, show=False)
    plt.tight_layout()
    path_dot = os.path.join(OUT_DIR, "shap_gb_summary_dot.png")
    plt.savefig(path_dot, dpi=200); plt.close()

    # bar summary
    plt.figure()
    shap.summary_plot(shap_values_gb, X_shap_gb, feature_names=feat_names_gb, plot_type="bar", show=False)
    plt.tight_layout()
    path_bar = os.path.join(OUT_DIR, "shap_gb_summary_bar.png")
    plt.savefig(path_bar, dpi=200); plt.close()

    print(f"Saved: {path_dot}\nSaved: {path_bar}")
except Exception as e:
    print("SHAP for GradientBoosting failed/skipped:", e)

# -----------------------------
# 4) SHAP for LogisticRegression (linear)
# -----------------------------
print("\nFitting LogisticRegression for SHAP/coefficients...")
lr = LogisticRegression(
    multi_class="multinomial",
    solver="saga",
    max_iter=5000,
    class_weight=class_weight_dict,
    n_jobs=-1
)
pipe_lr = Pipeline([("pre", pre), ("clf", lr)])
pipe_lr.fit(X, y)

# Transform for LR as well
X_trans_lr = pipe_lr.named_steps["pre"].transform(X)
feat_names_lr = pipe_lr.named_steps["pre"].get_feature_names_out()

# Try SHAP LinearExplainer (multiclass can work; if not, fall back to coefficients)
shap_ok = False
try:
    import shap, matplotlib.pyplot as plt
    # Background subset for linear explainer
    m = min(1000, X_trans_lr.shape[0])
    bg_idx = rng.choice(X_trans_lr.shape[0], size=m, replace=False)
    background = X_trans_lr[bg_idx]

    explainer_lr = shap.LinearExplainer(pipe_lr.named_steps["clf"], background, feature_dependence="independent")
    n_eval = min(1500, X_trans_lr.shape[0])
    eval_idx = rng.choice(X_trans_lr.shape[0], size=n_eval, replace=False)
    shap_values_lr = explainer_lr.shap_values(X_trans_lr[eval_idx])

    # If multiclass, shap_values_lr is a list; make a combined importance
    def mean_abs_shap(shap_vals):
        # shap_vals can be [n_classes x (n_samples x n_features)] or array
        if isinstance(shap_vals, list):
            # average |SHAP| across classes then across samples
            vals = np.mean([np.abs(sv).mean(axis=0) for sv in shap_vals], axis=0)
        else:
            vals = np.abs(shap_vals).mean(axis=0)
        return vals

    # summary (dot) for LR (use class 0 if list; SHAP needs an array)
    plt.figure()
    if isinstance(shap_values_lr, list):
        shap.summary_plot(shap_values_lr[0], X_trans_lr[eval_idx], feature_names=feat_names_lr, show=False)
    else:
        shap.summary_plot(shap_values_lr, X_trans_lr[eval_idx], feature_names=feat_names_lr, show=False)
    plt.tight_layout()
    path_lr_dot = os.path.join(OUT_DIR, "shap_lr_summary_dot.png")
    plt.savefig(path_lr_dot, dpi=200); plt.close()
    print(f"Saved: {path_lr_dot}")
    shap_ok = True

except Exception as e:
    print("SHAP for LogisticRegression failed; will export coefficients instead. Reason:", e)

# Fallback: export top coefficients (multinomial -> per-class)
if not shap_ok:
    clf = pipe_lr.named_steps["clf"]
    # coef_ shape: (n_classes, n_features)
    if hasattr(clf, "coef_"):
        coefs = pd.DataFrame(clf.coef_, columns=feat_names_lr, index=clf.classes_)
        # aggregate absolute weights across classes for ranking
        agg = coefs.abs().mean(axis=0).sort_values(ascending=False).head(50)
        out_coef = os.path.join(OUT_DIR, "lr_coefficients_top50.csv")
        agg.to_csv(out_coef, header=["mean_abs_weight"])
        print(f"Saved LR top coefficients: {out_coef}")
    else:
        print("LR has no coef_ attribute; cannot export coefficients.")

print("\nDone.")


Class weights: {'AD': 1.7520188425302827, 'CN': 0.8458414554905783, 'MCI': 0.8019405513630063}

Fitting GradientBoosting for SHAP...
SHAP for GradientBoosting failed/skipped: GradientBoostingClassifier is only supported for binary classification right now!

Fitting LogisticRegression for SHAP/coefficients...


C:\Users\acer\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


SHAP for LogisticRegression failed; will export coefficients instead. Reason: The option feature_dependence has been renamed to feature_perturbation!
Saved LR top coefficients: /mnt/data/model_compare_outputs\lr_coefficients_top50.csv

Done.


# HistGradientBoosting

In [3]:
# OPTIONAL: model-agnostic SHAP on your best model (HGB) via PermutationExplainer
import shap, numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(
    learning_rate=0.06, max_iter=400, max_depth=8, l2_regularization=0.5, random_state=RANDOM_STATE
)
pipe_hgb = Pipeline([("pre", pre), ("clf", hgb)])
pipe_hgb.fit(X, y)

X_hgb = pipe_hgb.named_steps["pre"].transform(X)
feat_hgb = pipe_hgb.named_steps["pre"].get_feature_names_out()

# (Optional) reduce to top-K features using the RF global ranking we saved
try:
    top25 = pd.read_csv(os.path.join(OUT_DIR, "shap_rf_top25_global.csv"), index_col=0).index.tolist()
    keep_mask = np.array([name in top25 for name in feat_hgb])
    X_hgb_small = X_hgb[:, keep_mask]
    feat_hgb_small = feat_hgb[keep_mask]
except Exception:
    X_hgb_small = X_hgb
    feat_hgb_small = feat_hgb

# Subsample rows
rng = np.random.RandomState(42)
n_rows = min(800, X_hgb_small.shape[0])  # keep smaller to speed up
row_idx = rng.choice(X_hgb_small.shape[0], size=n_rows, replace=False)
X_eval = X_hgb_small[row_idx]

# PermutationExplainer on predict_proba (multiclass OK)
f = lambda data: pipe_hgb.named_steps["clf"].predict_proba(data)
expl = shap.PermutationExplainer(f, X_eval, feature_names=feat_hgb_small)

max_evals = 2 * X_eval.shape[1] + 1
sv = expl(X_eval, max_evals=max_evals, silent=True)  # may take time

plt.figure()
shap.summary_plot(sv, X_eval, feature_names=feat_hgb_small, show=False)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "shap_hgb_summary_dot.png"), dpi=200); plt.close()
print("Saved HGB SHAP (Permutation) summary.")


C:\Users\acer\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\acer\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\acer\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 501, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\acer\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 969, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\acer\AppData\Local\Programs\Python\Python310\lib\subproces

Saved HGB SHAP (Permutation) summary.


In [7]:
# -*- coding: utf-8 -*-
import os, re, json
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.stats import loguniform, randint

from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier

# -----------------------------
# CONFIG
# -----------------------------
CSV_PATH     = "adni_merged_features.csv"      # your merged, labeled dataset
OUT_DIR      = "/mnt/data/randsearch_outputs"
RANDOM_STATE = 42
N_SPLITS     = 5
MAX_CAT_UNIQUES = 30

os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# 1) Load + basic checks
# -----------------------------
df = pd.read_csv(CSV_PATH, low_memory=False)
if "label" not in df.columns:
    raise SystemExit("No 'label' column found in the dataset.")
if "visit_id" not in df.columns:
    raise SystemExit("No 'visit_id' column found in the dataset.")

# Subject grouping: left part of visit_id (before first underscore)
groups = df["visit_id"].astype(str).str.split("_").str[0]
y = df["label"].astype(str)

# Remove obvious non-features
drop_cols = [c for c in ["label","visit_id","RID","PTID","ID","SITEID","USERDATE","update_stamp"] if c in df.columns]
X = df.drop(columns=drop_cols, errors="ignore")

# Defensive leakage filter (expandable)
EXCLUDE_PATTERNS = re.compile(
    r"(?:\bDIAGNOSIS\b|^DX|DXNORM|DXMCI|DXAD|DXAPP|DXCONFID|DXMOTHET|DXOTHDEM|DXDSEV"
    r"|HAS_QC_ERROR|SOURCE|DONE)$",
    re.I
)
leak_cols = [c for c in X.columns if EXCLUDE_PATTERNS.search(c)]
if leak_cols:
    print(f"[Leak filter] Dropping {len(leak_cols)} columns (examples: {leak_cols[:5]})")
    X = X.drop(columns=leak_cols, errors="ignore")

# Column type detection
cat_cols, num_cols = [], []
for c in X.columns:
    if pd.api.types.is_numeric_dtype(X[c]):
        num_cols.append(c)
    else:
        if X[c].nunique(dropna=True) <= MAX_CAT_UNIQUES:
            cat_cols.append(c)
        else:
            coerced = pd.to_numeric(X[c], errors="coerce")
            if coerced.notna().sum() >= 0.8*len(coerced):
                X[c] = coerced
                num_cols.append(c)
            else:
                X = X.drop(columns=[c])

# Rebuild lists after possible drops
all_cols = X.columns.tolist()
cat_cols = [c for c in cat_cols if c in all_cols]
num_cols = [c for c in num_cols if c in all_cols and c not in cat_cols]

# Preprocessor (fit inside CV to avoid leakage)
pre = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=True
)

# Class weights (balanced) for CN/MCI/AD
classes = np.sort(y.unique())
class_wts = compute_class_weight(class_weight="balanced", classes=classes, y=y)
CLASS_WEIGHT = {c: w for c, w in zip(classes, class_wts)}
print("Class weights:", CLASS_WEIGHT)

# -----------------------------
# 2) RandomizedSearchCV helpers
# -----------------------------
gkf = GroupKFold(n_splits=N_SPLITS)
macro_f1 = "f1_macro"  # primary scoring

def run_search(name, estimator, param_distributions, n_iter=40):
    pipe = Pipeline([("pre", pre), ("clf", estimator)])
    rs = RandomizedSearchCV(
        estimator=pipe,
        param_distributions={f"clf__{k}": v for k, v in param_distributions.items()},
        n_iter=n_iter,
        scoring=macro_f1,
        n_jobs=-1,
        cv=gkf.split(X, y, groups),
        verbose=1,
        random_state=RANDOM_STATE,
        refit=True,
        return_train_score=False
    )
    rs.fit(X, y)
    print(f"\n=== {name}: RandomizedSearchCV ===")
    print("Best params:", rs.best_params_)
    print("Best CV macro_f1:", rs.best_score_)
    # Evaluate on refit (all data) for a quick sanity check
    yhat = rs.best_estimator_.predict(X)
    rep = classification_report(y, yhat, zero_division=0)
    print("\nRefit-on-all classification report (not OOF):\n", rep)

    # Save CV table
    cv_tbl = pd.DataFrame(rs.cv_results_).sort_values("mean_test_score", ascending=False)
    cv_tbl.to_csv(os.path.join(OUT_DIR, f"{name}_randsearch_results.csv"), index=False)
    return rs.best_estimator_, rs.best_params_, rs.best_score_

# -----------------------------
# 3) Define search spaces
# -----------------------------
space_lr = {
    # LogisticRegression: use saga for multinomial + class_weight dict
    "C": loguniform(1e-3, 1e+1),
    "penalty": ["l2"],        # keep simple/stable
    # "l1_ratio": [0.0, 0.25, 0.5]  # (only if penalty='elasticnet')
}
space_rf = {
    "n_estimators": randint(300, 900),
    "max_depth": [None, 10, 12, 16, 20],
    "min_samples_leaf": randint(1, 5),
    "max_features": ["sqrt", 0.5, 0.8]
}
space_gb = {
    "learning_rate": loguniform(1e-3, 2e-1),
    "n_estimators": randint(200, 800),
    "max_depth": randint(2, 5),
    "subsample": [0.7, 0.85, 1.0]
}
space_hgb = {
    "learning_rate": loguniform(1e-3, 2e-1),
    "max_depth": [None, 6, 8, 12],
    "max_leaf_nodes": [31, 63, 127],
    "l2_regularization": loguniform(1e-4, 1e-1),
    "min_samples_leaf": randint(10, 60),
    "early_stopping": [True],
    "validation_fraction": [0.1, 0.2]
}

# -----------------------------
# 4) Run searches
# -----------------------------
best_lr = run_search(
    "LogReg",
    LogisticRegression(
        multi_class="multinomial", solver="saga", max_iter=5000,
        class_weight=CLASS_WEIGHT, random_state=RANDOM_STATE
    ),
    space_lr, n_iter=40
)

best_rf = run_search(
    "RF",
    RandomForestClassifier(
        class_weight=CLASS_WEIGHT, random_state=RANDOM_STATE, n_jobs=-1
    ),
    space_rf, n_iter=60
)

best_gb = run_search(
    "GB",
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    space_gb, n_iter=50
)

best_hgb = run_search(
    "HGB",
    HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    space_hgb, n_iter=60
)

print("\nDone. RandomizedSearchCV results saved in:", OUT_DIR)


[Leak filter] Dropping 27 columns (examples: ['DXSUM__DXNORM', 'DXSUM__DXMCI', 'DXSUM__DXMOTHET', 'DXSUM__DXDSEV', 'DXSUM__DXAD'])
Class weights: {'AD': 1.7193808882907133, 'CN': 0.8300844704353476, 'MCI': 0.7870013861081164, 'nan': 1.060385972193401}
Fitting 5 folds for each of 40 candidates, totalling 200 fits


KeyboardInterrupt: 

In [ ]:
# -*- coding: utf-8 -*-
import os, re
import numpy as np
import pandas as pd

from pathlib import Path
from joblib import Memory

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import HalvingGridSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier

# -----------------------------
# CONFIG
# -----------------------------
CSV_PATH     = "adni_merged_features.csv"
OUT_DIR      = "/mnt/data/gridsearch_outputs_fast"
RANDOM_STATE = 42
N_SPLITS     = 5
MAX_CAT_UNIQUES = 30

os.makedirs(OUT_DIR, exist_ok=True)
memory = Memory(location=os.path.join(OUT_DIR, "pipeline_cache"), verbose=0)

# -----------------------------
# Load + basic checks
# -----------------------------
df = pd.read_csv(CSV_PATH, low_memory=False)
if "label" not in df.columns:
    raise SystemExit("No 'label' column found in the dataset.")
if "visit_id" not in df.columns:
    raise SystemExit("No 'visit_id' column found in the dataset.")

y = df["label"].astype(str)
groups = df["visit_id"].astype(str).str.split("_").str[0]

drop_cols = [c for c in ["label","visit_id","RID","PTID","ID","SITEID","USERDATE","update_stamp"] if c in df.columns]
X = df.drop(columns=drop_cols, errors="ignore")

# Defensive leak filter
EXCLUDE_PATTERNS = re.compile(
    r"(?:\bDIAGNOSIS\b|^DX|DXNORM|DXMCI|DXAD|DXAPP|DXCONFID|DXMOTHET|DXOTHDEM|DXDSEV"
    r"|HAS_QC_ERROR|SOURCE|DONE)$",
    re.I
)
leak_cols = [c for c in X.columns if EXCLUDE_PATTERNS.search(c)]
if leak_cols:
    print(f"[Leak filter] Dropping {len(leak_cols)} columns (examples: {leak_cols[:6]})")
    X = X.drop(columns=leak_cols, errors="ignore")

# Type detection
cat_cols, num_cols = [], []
for c in X.columns:
    if pd.api.types.is_numeric_dtype(X[c]):
        num_cols.append(c)
    else:
        if X[c].nunique(dropna=True) <= MAX_CAT_UNIQUES:
            cat_cols.append(c)
        else:
            coerced = pd.to_numeric(X[c], errors="coerce")
            if coerced.notna().sum() >= 0.8*len(coerced):
                X[c] = coerced
                num_cols.append(c)
            else:
                X = X.drop(columns=[c])

# Recompute after drops
all_cols = X.columns.tolist()
cat_cols = [c for c in cat_cols if c in all_cols]
num_cols = [c for c in num_cols if c in all_cols and c not in cat_cols]

# Preprocessor (include scaling only for numeric, trees will ignore)
pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc",  StandardScaler(with_mean=False))  # sparse-safe + speeds LR
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh",  OneHotEncoder(handle_unknown="ignore", sparse_output=True))
        ]), cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=True
)

# Class weights
classes = np.sort(y.unique())
class_wts = compute_class_weight(class_weight="balanced", classes=classes, y=y)
CLASS_WEIGHT = {c: w for c, w in zip(classes, class_wts)}
print("Class weights:", CLASS_WEIGHT)

# CV splits (precompute once)
gkf = GroupKFold(n_splits=N_SPLITS)
cv_splits = list(gkf.split(X, y, groups))

# Base pipelines (cached)
pipe_lr  = Pipeline([("pre", pre), ("clf", LogisticRegression(multi_class="multinomial", solver="saga",
                                                             class_weight=CLASS_WEIGHT, random_state=RANDOM_STATE,
                                                             max_iter=2000, tol=1e-3))], memory=memory)
pipe_rf  = Pipeline([("pre", pre), ("clf", RandomForestClassifier(class_weight=CLASS_WEIGHT,
                                                                  random_state=RANDOM_STATE, n_jobs=-1))], memory=memory)
pipe_gb  = Pipeline([("pre", pre), ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))], memory=memory)
pipe_hgb = Pipeline([("pre", pre), ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE))], memory=memory)

# **Tight grids** (keeps runtime small but useful)
grid_lr = {
    "clf__C": [0.05, 0.1, 0.2, 0.5, 1.0],
    # 'saga' + 'l2' only (stable). Higher tol & capped max_iter already in estimator.
}
grid_rf = {
    "clf__n_estimators": [300, 500],
    "clf__max_depth": [None, 14, 18],
    "clf__min_samples_leaf": [1, 2, 4],
    "clf__max_features": ["sqrt", 0.8]
}
grid_gb = {
    "clf__learning_rate": [0.03, 0.06, 0.1],
    "clf__n_estimators": [300, 500],
    "clf__max_depth": [2, 3],
    "clf__subsample": [0.7, 0.85, 1.0]
}
grid_hgb = {
    "clf__learning_rate": [0.03, 0.06, 0.1],
    "clf__max_depth": [None, 8],
    "clf__max_leaf_nodes": [31, 63],
    "clf__l2_regularization": [1e-3, 3e-3, 1e-2],
    "clf__min_samples_leaf": [20, 40],
    "clf__early_stopping": [True],
    "clf__validation_fraction": [0.1]
}

# For LogisticRegression use Successive Halving (cuts off weak combos early)
gs_lr = HalvingGridSearchCV(
    estimator=pipe_lr,
    param_grid=grid_lr,
    factor=3,
    scoring="f1_macro",
    cv=cv_splits,
    n_jobs=-1,
    verbose=1,
    refit=False,
    error_score="raise"
)

# For trees/boosting: compact GridSearchCV
gs_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=grid_rf,
    scoring="f1_macro",
    cv=cv_splits,
    n_jobs=-1,
    verbose=1,
    refit=False,
)

gs_gb = GridSearchCV(
    estimator=pipe_gb,
    param_grid=grid_gb,
    scoring="f1_macro",
    cv=cv_splits,
    n_jobs=-1,
    verbose=1,
    refit=False,
)

gs_hgb = GridSearchCV(
    estimator=pipe_hgb,
    param_grid=grid_hgb,
    scoring="f1_macro",
    cv=cv_splits,
    n_jobs=-1,
    verbose=1,
    refit=False,
)

searches = [
    ("LogisticRegression (HalvingGrid)", gs_lr),
    ("RandomForest (Grid)", gs_rf),
    ("GradientBoosting (Grid)", gs_gb),
    ("HistGradientBoosting (Grid)", gs_hgb),
]

best_models = []
for name, gs in searches:
    print(f"\n===== {name} =====")
    gs.fit(X, y)  # uses precomputed cv_splits
    # pick best params & refit once on full data for reporting
    best_params = gs.best_params_
    print("Best params:", best_params)

    # Build a fresh pipeline with those params and fit once
    base = gs.estimator
    best = Pipeline([("pre", pre), ("clf", base.named_steps["clf"].__class__(**{
        k.replace("clf__", ""): v for k, v in best_params.items()
    }))], memory=memory)
    best.fit(X, y)
    yhat = best.predict(X)
    print("\nRefit-on-all report (not OOF):\n", classification_report(y, yhat, zero_division=0))
    best_models.append((name, best, best_params))

# Optional: save a tiny summary table
rows = []
for name, _, params in best_models:
    row = {"model": name}
    row.update({k.replace("clf__", ""): v for k, v in params.items()})
    rows.append(row)
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "best_params_summary.csv"), index=False)

print("\nSaved:", os.path.join(OUT_DIR, "best_params_summary.csv"))
print("Done.")


In [1]:
# -*- coding: utf-8 -*-
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, make_scorer
# Assuming your previous data loading and preprocessing steps (Sections 1-4) have been executed:
# X, y, groups, pre, class_weight_dict, N_SPLITS, RANDOM_STATE, classes

# --- [ 1-4: Data Loading, Preprocessing (pre), and Groups - Assuming these are done ] ---
# Replace these with your actual loaded data and preprocessor 'pre'
# X = ...
# y = ...
# groups = ...
# pre = ... (Your ColumnTransformer)
# class_weight_dict = ...
# N_SPLITS = 5
# RANDOM_STATE = 42

# ----------------------------------------------------
# 5) Define Model, Parameter Grid, and Search Strategy
# ----------------------------------------------------

# 5.1) Define the Model Pipeline
# Note: The 'pre' object is the ColumnTransformer you defined previously.
logreg_pipeline = Pipeline(steps=[
    ('pre', pre),
    ('clf', LogisticRegression(
        multi_class="multinomial",
        max_iter=2000,
        class_weight=class_weight_dict,
        random_state=RANDOM_STATE
    ))
])

# 5.2) Define the Parameter Grid for Logistic Regression
# Parameters are referenced using the step name ('clf') followed by a double underscore ('__')
# and the parameter name (e.g., 'C').
param_grid_logreg = {
    # Regularization strength: C=1.0 is default, lower is stronger regularization
    'clf__C': [0.1, 1.0, 10.0, 100.0],
    
    # Algorithm to use in the optimization problem
    'clf__solver': ['lbfgs', 'newton-cg'] # Suitable solvers for multi_class='multinomial'
}

# 5.3) Define the Scoring Metric
# Use macro F1-score as the primary metric for optimization, as it handles multi-class and imbalance.
scorer = make_scorer(f1_score, average='macro')

# 5.4) Define the Cross-Validation Strategy (GroupKFold)
# This ensures that all visits from the same subject (groups) stay in the same split.
gkf = GroupKFold(n_splits=N_SPLITS)

# 5.5) Implement GridSearchCV
print("\n===== Starting GridSearchCV for Logistic Regression =====")
grid_search = GridSearchCV(
    estimator=logreg_pipeline,        # The pipeline to tune
    param_grid=param_grid_logreg,     # The dictionary of parameters to search
    scoring=scorer,                   # The metric to optimize (F1 Macro)
    cv=gkf.split(X, y, groups),       # The cross-validation splits (must use the groups)
    verbose=3,                        # Higher verbosity shows more progress
    n_jobs=-1                         # Use all available CPU cores
)

# ----------------------------------------------------
# 6) Execute the Search
# ----------------------------------------------------

# Note: The groups array MUST be passed again to the fit method for GroupKFold to work correctly.
grid_search.fit(X, y, groups=groups)

# ----------------------------------------------------
# 7) Results and Summary
# ----------------------------------------------------

print("\n\n=== GridSearchCV Results ===")

# The best combination of parameters found
print(f"Best parameters found: {grid_search.best_params_}")

# The mean score (macro F1) achieved with the best parameters
print(f"Best Macro F1 Score (CV): {grid_search.best_score_:.4f}")

# The best model object (now fitted on all data with the best parameters)
best_model = grid_search.best_estimator_

# Optional: View the full results (all parameter combinations and their scores)
results_df = pd.DataFrame(grid_search.cv_results_).sort_values('rank_test_score')
print("\nTop 5 Parameter Combinations:")
print(results_df[['param_clf__C', 'param_clf__solver', 'mean_test_score', 'rank_test_score']].head())

# To export the best model for future use:
# from joblib import dump
# dump(best_model, 'best_logreg_model.joblib')

NameError: name 'pre' is not defined

In [ ]:
# -*- coding: utf-8 -*-
import os, re
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, make_scorer
from sklearn.utils.class_weight import compute_class_weight

from sklearn.linear_model import LogisticRegression

# ----------------------------------------------------
# CONFIG
# ----------------------------------------------------
# !!! CHANGE THIS PATH IF YOUR FILE IS ELSEWHERE !!!
CSV_PATH = "adni_merged_features.csv" 
RANDOM_STATE = 42
N_SPLITS = 5
MAX_CAT_UNIQUES = 30  # columns with <=30 unique values (non-numeric) treated as categoricals

# ----------------------------------------------------
# 1) Load data
# ----------------------------------------------------
try:
    df = pd.read_csv(CSV_PATH, low_memory=False)
except FileNotFoundError:
    print(f"Error: CSV file not found at {CSV_PATH}. Please check the path.")
    exit()

if "label" not in df.columns:
    raise SystemExit("No 'label' column found in the dataset.")

# Robust subject groups: attempt to extract RID from 'visit_id' prefix
if "visit_id" not in df.columns:
    raise SystemExit("No 'visit_id' column found. Needed to group by subject.")

# Groups are used for GroupKFold to separate subjects
groups = df["visit_id"].astype(str).str.split("_").str[0]

# Drop non-feature identifier columns
drop_cols = [c for c in ["visit_id","RID","PTID","ID","SITEID","USERDATE","update_stamp"] if c in df.columns]
X = df.drop(columns=["label"] + drop_cols, errors="ignore")
y = df["label"].astype(str)

# ----------------------------------------------------
# 2) Identify column types + defensive leak filtering
# ----------------------------------------------------
# Exclude any diagnosis-like or QC/meta columns that might have slipped in
EXCLUDE_PATTERNS = re.compile(r"(DIAGNOSIS|^DX|DXNORM|DXMCI|DXAD|DXAPP|CONFID|MOTHET|OTHDEM|DSEV|HAS_QC_ERROR|SOURCE|DONE)$", re.I)

leak_cols = [c for c in X.columns if EXCLUDE_PATTERNS.search(c)]
if leak_cols:
    X = X.drop(columns=leak_cols, errors="ignore")

# Decide categorical vs numeric
cat_cols, num_cols = [], []
for c in X.columns:
    if pd.api.types.is_numeric_dtype(X[c]):
        num_cols.append(c)
    else:
        # treat as categorical if low-cardinality non-numeric
        if X[c].nunique(dropna=True) <= MAX_CAT_UNIQUES:
            cat_cols.append(c)
        else:
            # Try coercion to numeric; if fails, drop as high-card text
            coerced = pd.to_numeric(X[c], errors="coerce")
            if coerced.notna().sum() >= 0.8 * len(coerced):
                X[c] = coerced
                num_cols.append(c)
            else:
                # too messy -> drop
                X = X.drop(columns=[c])

# Recompute final lists after possible drops
all_cols = X.columns.tolist()
cat_cols = [c for c in cat_cols if c in all_cols]
num_cols = [c for c in num_cols if c in all_cols and c not in cat_cols]
print(f"Features: {len(num_cols)} numeric, {len(cat_cols)} categorical.")

# ----------------------------------------------------
# 3) Preprocessor (ColumnTransformer: Impute + Encode)
# ----------------------------------------------------
pre = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline(steps=[
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=True
)

# ----------------------------------------------------
# 4) Class weights (to balance CN/MCI/AD)
# ----------------------------------------------------
classes = np.sort(y.unique())
class_wts = compute_class_weight(class_weight="balanced", classes=classes, y=y)
class_weight_dict = {cls: wt for cls, wt in zip(classes, class_wts)}
print("Class weights:", class_weight_dict)

# ----------------------------------------------------
# 5) GridSearchCV Implementation (Tuning Logistic Regression)
# ----------------------------------------------------

# 5.1) Define the Model Pipeline
logreg_pipeline = Pipeline(steps=[
    ('pre', pre),
    ('clf', LogisticRegression(
        multi_class="multinomial",
        max_iter=2000,
        class_weight=class_weight_dict,
        random_state=RANDOM_STATE
    ))
])

# 5.2) Define the Parameter Grid
param_grid_logreg = {
    # C is the inverse of regularization strength (smaller C -> stronger regularization)
    'clf__C': [0.1, 1.0, 10.0, 100.0],
    
    # Solvers compatible with multi_class='multinomial'
    'clf__solver': ['lbfgs', 'newton-cg'] 
}

# 5.3) Define the Scoring Metric
# Use macro F1-score to optimize for overall balance across classes
scorer = make_scorer(f1_score, average='macro')

# 5.4) Define the Cross-Validation Strategy (GroupKFold)
gkf = GroupKFold(n_splits=N_SPLITS)

# 5.5) Implement GridSearchCV
print("\n===== Starting GridSearchCV for Logistic Regression =====")
grid_search = GridSearchCV(
    estimator=logreg_pipeline,        # The pipeline to tune
    param_grid=param_grid_logreg,     # The parameter combinations
    scoring=scorer,                   # Metric to optimize
    # Pass GroupKFold splits, ensuring subject separation
    cv=gkf,                           
    verbose=3,                        
    n_jobs=-1                         # Use all available CPU cores
)

# ----------------------------------------------------
# 6) Execute the Search
# ----------------------------------------------------

# Note: The groups array MUST be passed to the fit method for GroupKFold to work.
grid_search.fit(X, y, groups=groups)

# ----------------------------------------------------
# 7) Results and Summary
# ----------------------------------------------------

print("\n\n=== GridSearchCV Final Results ===")

# The best combination of parameters found
print(f"Best parameters found: {grid_search.best_params_}")

# The mean macro F1 score achieved with the best parameters
print(f"Best Macro F1 Score (CV): {grid_search.best_score_:.4f}")

# The best model object (fitted on all data with the best parameters)
best_model = grid_search.best_estimator_

# Optional: View the full results
results_df = pd.DataFrame(grid_search.cv_results_).sort_values('rank_test_score')
print("\nTop Parameter Combinations:")
print(results_df[['param_clf__C', 'param_clf__solver', 'mean_test_score', 'rank_test_score']].head())

Features: 321 numeric, 0 categorical.
Class weights: {'AD': 1.7193808882907133, 'CN': 0.8300844704353476, 'MCI': 0.7870013861081164, 'nan': 1.060385972193401}

===== Starting GridSearchCV for Logistic Regression =====
Fitting 5 folds for each of 8 candidates, totalling 40 fits
